# Concatenation Test
Concatenate clips within a gt:severity:speaker group into a single stream.
Listen for transitions between clips.

In [1]:
import os
import torchaudio
import torch

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
MISC_DIR = os.path.join(BASE_DIR, "data", "misc")
os.makedirs(MISC_DIR, exist_ok=True)
TARGET_SR = 16000

# Three example groups
groups = {
    "control_gt0_spk47": [
        "data/control/ctrl_gt0_spk47_s000.wav",
        "data/control/ctrl_gt0_spk47_s001.wav",
        "data/control/ctrl_gt0_spk47_s002.wav",
        "data/control/ctrl_gt0_spk47_s003.wav",
        "data/control/ctrl_gt0_spk47_s004.wav",
    ],
    "moderate_gt100_spk28": [
        "data/dysfluent/moderate/dys_moderate_gt100_spk28_s000.wav",
        "data/dysfluent/moderate/dys_moderate_gt100_spk28_s001.wav",
        "data/dysfluent/moderate/dys_moderate_gt100_spk28_s002.wav",
        "data/dysfluent/moderate/dys_moderate_gt100_spk28_s003.wav",
    ],
    "severe_gt101_spk107": [
        "data/dysfluent/severe/dys_severe_gt101_spk107_s000.wav",
        "data/dysfluent/severe/dys_severe_gt101_spk107_s001.wav",
        "data/dysfluent/severe/dys_severe_gt101_spk107_s002.wav",
        "data/dysfluent/severe/dys_severe_gt101_spk107_s003.wav",
    ],
}

# Crossfade duration in samples
CROSSFADE_MS = 50
CROSSFADE_SAMPLES = int(TARGET_SR * CROSSFADE_MS / 1000)

for name, files in groups.items():
    clips = []
    for f in files:
        wav, sr = torchaudio.load(os.path.join(BASE_DIR, f))
        if sr != TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
        clips.append(wav.squeeze(0))
    
    # Concatenate with short crossfade to smooth transitions
    concat = clips[0]
    for clip in clips[1:]:
        n = min(CROSSFADE_SAMPLES, len(concat), len(clip))
        fade_out = torch.linspace(1, 0, n)
        fade_in = torch.linspace(0, 1, n)
        concat[-n:] = concat[-n:] * fade_out + clip[:n] * fade_in
        concat = torch.cat([concat, clip[n:]])
    
    out_path = os.path.join(MISC_DIR, f"concat_{name}.wav")
    torchaudio.save(out_path, concat.unsqueeze(0), TARGET_SR)
    print(f"{name}: {len(clips)} clips → {concat.shape[0]/TARGET_SR:.1f}s → {out_path}")

control_gt0_spk47: 5 clips → 31.7s → /data/liharrison/lvsim/data/misc/concat_control_gt0_spk47.wav
moderate_gt100_spk28: 4 clips → 50.1s → /data/liharrison/lvsim/data/misc/concat_moderate_gt100_spk28.wav
severe_gt101_spk107: 4 clips → 42.0s → /data/liharrison/lvsim/data/misc/concat_severe_gt101_spk107.wav
